# Task-aware compression of R2Gen features: code walkthrough

**Task (from the technical note):** spend fewer bits while preserving the clinically meaningful content of the
report generated from the compressed feature.

**Setting.** R2Gen (ResNet-101 + Transformer with relational memory) is pretrained on MIMIC-CXR and **frozen**.
Only a learned codec between its image encoder and its report decoder is trained:

```
X-ray --[ResNet-101, frozen]--> feature 2048x7x7 (3,211,264 bits) --[codec: encode]--> 500-2,000 bits
      --[codec: decode]--> approximate feature 2048x7x7 --[Transformer, frozen]--> report
```

Two codecs are compared at the same actual bit rate:

| name | training objective |
|---|---|
| **Bft** (feature-only) | rate + feature reconstruction error |
| **F** (task-aware) | rate + feature error + **span-weighted NLL of the uncompressed model's own report** + **token-level KL to the uncompressed model** |

The radiologist's report is never used to train F: its target is the report R2Gen writes from the
*uncompressed* feature, so a perfect codec scores zero loss and no codec can "beat" the uncompressed model.

**What this notebook does.** It shows the code of every part (the codec class and the losses are written out
in full; data loading and R2Gen wiring are imported from `R2Gen/task_aware/`), runs one test image through the
whole pipeline, decodes reports at 500/1,000/2,000 bits, runs a few real training steps, and rebuilds the
result tables and charts from the saved evaluations. **Nothing is retrained or overwritten**; trained codecs are
loaded from `results/mimic_full/`.

**Requirements.** The project's Python environment (the one used for `run_mimic.sh`: PyTorch, CompressAI 1.2.8,
torchvision) plus `notebook`/`ipykernel`. A GPU makes it run in a few minutes; on CPU the training demo uses
fewer steps. Open the notebook from the `medical-compression` folder.

> MIMIC-CXR data and the report text printed below fall under the PhysioNet data use agreement:
> share this notebook only with credentialed users and do not upload it publicly.

## 0. Setup
Finds the project folders and imports the project modules: `ta_common` (R2Gen loading, cached features,
targets), `clinical_spans` (rule-based clinical labeller) and `feature_codec` (the codec, used here only to
cross-check the copy written out in Section 3).

In [ ]:
import os, sys, json, math, time, inspect
from pathlib import Path

ROOT = Path.cwd().resolve()
for cand in [ROOT, *ROOT.parents]:
    if (cand / 'R2Gen' / 'task_aware').is_dir() and (cand / 'results').is_dir():
        ROOT = cand
        break
else:
    raise FileNotFoundError('Open this notebook from the medical-compression folder.')
R2GEN = ROOT / 'R2Gen'
TA = R2GEN / 'task_aware'
FEAT = ROOT / 'results' / 'mimic_features'      # cached ResNet layer4 features (train/val/test)
RESULTS = ROOT / 'results' / 'mimic_full'        # trained codecs, teacher reports, evaluations
os.chdir(R2GEN)                                  # R2Gen code uses paths relative to this folder
for p in (str(R2GEN), str(TA)):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import compressai
from compressai.entropy_models import EntropyBottleneck, GaussianConditional

import ta_common as tc          # frozen R2Gen wrapper, FeatureStore, TargetBank
import clinical_spans as cs     # rule-based clinical spans / labels (ta-spans-1.0)
import feature_codec as fcodec  # project codec module (cross-check + bitstream container)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('project   :', ROOT)
print('torch     :', torch.__version__, '| compressai', compressai.__version__, '| device:', device)

## 1. The frozen R2Gen model
Same arguments as `task_aware/run_mimic.sh` (beam 3, trigram blocking, max 100 tokens). `load_frozen_r2gen`
loads the public MIMIC-CXR checkpoint, switches to eval mode and sets `requires_grad=False` on every
parameter. Gradients can still flow *through* the frozen decoder into the codec during training.

In [ ]:
R2GEN_ARGV = ['--image_dir', 'data/mimic_cxr/images/', '--ann_path', 'data/mimic_cxr/annotation.json',
              '--dataset_name', 'mimic_cxr', '--load', 'data/model_mimic_cxr.pth', '--num_workers', '0',
              '--beam_size', '3', '--sample_method', 'beam_search', '--block_trigrams', '1',
              '--max_seq_length', '100', '--threshold', '10']
args = tc.r2gen_args(R2GEN_ARGV)
tc.fix_seed(args.seed)
r2, tokenizer = tc.load_frozen_r2gen(args, device)

def n_params(module):
    return sum(p.numel() for p in module.parameters())

print(f'ResNet-101 visual extractor : {n_params(r2.model.visual_extractor)/1e6:6.1f} M parameters')
print(f'Transformer encoder-decoder : {n_params(r2.model.encoder_decoder)/1e6:6.1f} M parameters')
print('decoder config:', {k: getattr(args, k) for k in ('d_model', 'd_ff', 'num_layers', 'num_heads',
                                                       'rm_num_slots', 'rm_d_model')})
print('vocabulary   :', tokenizer.get_vocab_size(), 'tokens')
print('trainable R2Gen parameters:', sum(p.numel() for p in r2.model.parameters() if p.requires_grad))

## 2. One test image through the uncompressed pipeline
Features of the 12,000 / 1,000 / 3,858 train / val / test images were extracted once
(`extract_features.py`) and cached as float32 arrays of shape `[N, 1, 2048, 7, 7]` (one view per MIMIC image).

The example image is picked automatically from the saved 500-bit test evaluation: a study where the uncompressed
model reports an abnormal finding, the task-aware codec F keeps the same findings and the feature-only codec Bft
does not. Set `EX_ID` by hand to look at any other test study.

In [ ]:
ann = tc.load_annotation(args)
ann_test = {e['id']: e for e in ann['test']}
test_store = tc.FeatureStore(FEAT, 'test')
print('cached test features:', test_store.features.shape, test_store.features.dtype)

saved500 = json.loads((RESULTS / 'eval_test_r500' / 'per_study.json').read_text(encoding='utf-8'))
saved500 = {c: {r['id']: r for r in saved500[c]} for c in ('A', 'Bft', 'F')}

def abnormal(labels):
    return {c for c, s in labels.items() if s in ('positive', 'uncertain')}

candidates = [i for i in test_store.ids
              if abnormal(saved500['A'][i]['labels'])
              and abnormal(saved500['F'][i]['labels']) == abnormal(saved500['A'][i]['labels'])
              and abnormal(saved500['Bft'][i]['labels']) != abnormal(saved500['A'][i]['labels'])]
print(f'{len(candidates)} test studies where F keeps the uncompressed findings at 500 bits and Bft does not')
EX_ID = candidates[0]
EX = test_store.ids.index(EX_ID)
ex = ann_test[EX_ID]
print('example:', EX_ID, '| study', ex['study_id'])

In [ ]:
from PIL import Image
from torchvision import transforms

img = Image.open(os.path.join(args.image_dir, ex['image_path'][0])).convert('RGB')
plt.figure(figsize=(4, 4)); plt.imshow(img); plt.axis('off')
plt.title(f"MIMIC-CXR test study {ex['study_id']}"); plt.show()
print('Radiologist report (reference, never used to train F):\n')
print(tokenizer.clean_report(ex['report']))

**Stage 1: ResNet-101 ("eyes").** The evaluation transform is Resize(224,224) + ImageNet normalisation. The output
of the last convolutional block (layer4) is a 7x7 grid with 2,048 numbers per cell. The live feature should match
the cached one up to GPU rounding.

In [ ]:
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
x_img = tf(img).unsqueeze(0).to(device)               # [1, 3, 224, 224]
f_live = r2.cnn_features(x_img)[0]                    # [1, 2048, 7, 7]
x_study = test_store.batch([EX], device)              # [1, 1, 2048, 7, 7]  (batch, views, C, H, W)
f_cached = x_study[:, 0]

print('image tensor  :', tuple(x_img.shape))
print('layer4 feature:', tuple(f_live.shape))
print(f'max |live - cached| = {(f_live - f_cached).abs().max().item():.2e}')
RAW_BITS = f_cached.numel() * 32
print(f'uncompressed feature: {f_cached.numel():,} float32 values = {RAW_BITS:,} bits')

**Stage 3: Transformer decoder ("writer").** R2Gen turns the 7x7 grid into 49 "patch" vectors (`att`, used by
attention) and one average vector (`fc`), then writes the report token by token with beam search. This
uncompressed report (condition **A**) is the reference point for everything below and the training target of F.

In [ ]:
fc, att = r2.to_r2gen_inputs(f_cached)
print('fc :', tuple(fc.shape), ' (mean over the 7x7 grid)')
print('att:', tuple(att.shape), ' (49 patch vectors)')

report_A = r2.generate(f_cached, None)[0]
print('\nUncompressed R2Gen report (A):\n', report_A)
print('\nabnormal findings (rule labeller):', sorted(abnormal(cs.report_labels(report_A.split()))))
print('same text as the saved evaluation:', report_A == saved500['A'][EX_ID]['generated'])

## 3. The codec
A mean-scale hyperprior (Balle et al. 2018; Minnen et al. 2018) adapted from images to a 7x7 feature map. The
entropy models are CompressAI's `EntropyBottleneck` and `GaussianConditional`. The architecture is standard;
the contribution of the project is the training objective in Section 5.

| block | layers | output for one image |
|---|---|---|
| normalise | per-channel (x - mean) / std, statistics from the training split | 2048 x 7 x 7 |
| `g_a` analysis | conv3x3 2048->512, GELU, conv3x3 512->64 | **y**: 64 x 7 x 7 |
| `h_a` hyper-analysis | conv3x3 stride 2 64->32, GELU, conv3x3 stride 2 32->32 | **z**: 32 x 2 x 2 |
| `EntropyBottleneck` | learned factorised density for z, rounds and codes z | z bits |
| `h_s` hyper-synthesis | convT 4x4 x2 32->32, GELU, convT 4x4 x2 32->128, crop to 7x7 | mean and scale of each y |
| `GaussianConditional` | rounds y, codes it with N(mean, scale) | y bits |
| `g_s` synthesis | conv3x3 64->512, GELU, conv1x1 512->2048, de-normalise | x_hat: 2048 x 7 x 7 |

Training uses additive uniform noise instead of rounding (differentiable). Evaluation rounds, and
`mode='round'` gives exactly the same reconstruction as real `compress()` -> bytes -> `decompress()`.

The class below is a copy of `FeatureHyperpriorCodec` from `task_aware/feature_codec.py`.

In [ ]:
SCALES_MIN, SCALES_MAX, SCALES_LEVELS = 0.11, 256.0, 64


def get_scale_table(lo=SCALES_MIN, hi=SCALES_MAX, levels=SCALES_LEVELS):
    return torch.exp(torch.linspace(math.log(lo), math.log(hi), levels))


def ste_round(x):
    return (torch.round(x) - x).detach() + x


def _z_hw(h, w):
    # two stride-2 convs with k=3, p=1: n -> ceil(n/2) -> ceil(ceil(n/2)/2)   (7 -> 4 -> 2)
    return ((h + 1) // 2 + 1) // 2, ((w + 1) // 2 + 1) // 2


class FeatureHyperpriorCodec(nn.Module):
    def __init__(self, in_channels=2048, hidden=512, c_y=128, c_z=None):
        super().__init__()
        c_h = max(c_y // 2, 32)
        c_z = c_z or c_h
        self.config = {'in_channels': in_channels, 'hidden': hidden, 'c_y': c_y, 'c_z': c_z}
        self.c_y, self.c_z = c_y, c_z
        self.register_buffer('norm_mean', torch.zeros(1, in_channels, 1, 1))
        self.register_buffer('norm_std', torch.ones(1, in_channels, 1, 1))
        self.g_a = nn.Sequential(nn.Conv2d(in_channels, hidden, 3, 1, 1), nn.GELU(),
                                 nn.Conv2d(hidden, c_y, 3, 1, 1))
        self.h_a = nn.Sequential(nn.Conv2d(c_y, c_h, 3, 2, 1), nn.GELU(),
                                 nn.Conv2d(c_h, c_z, 3, 2, 1))
        self.h_s = nn.Sequential(nn.ConvTranspose2d(c_z, c_h, 4, 2, 1), nn.GELU(),
                                 nn.ConvTranspose2d(c_h, 2 * c_y, 4, 2, 1))
        self.g_s = nn.Sequential(nn.Conv2d(c_y, hidden, 3, 1, 1), nn.GELU(),
                                 nn.Conv2d(hidden, in_channels, 1))
        self.entropy_bottleneck = EntropyBottleneck(c_z)
        self.gaussian_conditional = GaussianConditional(None)

    # ------------------------------------------------------------------ utils
    @torch.no_grad()
    def set_normalization(self, mean, std, eps=1e-6):
        self.norm_mean.copy_(mean.reshape(self.norm_mean.shape))
        self.norm_std.copy_(std.reshape(self.norm_std.shape).clamp_min(eps))

    def trainable_groups(self):
        # main (g_a, g_s, h_a, h_s, entropy densities) vs aux (EntropyBottleneck quantiles)
        main, aux = [], []
        for n, p in self.named_parameters():
            (aux if n.endswith('.quantiles') else main).append(p)
        return main, aux

    def aux_loss(self):
        return self.entropy_bottleneck.loss()

    def update(self, force=True):
        self.gaussian_conditional.update_scale_table(get_scale_table(), force=force)
        self.entropy_bottleneck.update(force=force)

    def _hyper_params(self, z_hat, hw):
        p = self.h_s(z_hat)[..., :hw[0], :hw[1]]          # 8x8 -> crop to 7x7
        if p.shape[-2:] != torch.Size(hw):
            raise ValueError(f'h_s output {tuple(p.shape[-2:])} smaller than latent grid {hw}')
        means, scales = p.chunk(2, dim=1)
        return means, F.softplus(scales)

    # ---------------------------------------------------------------- forward
    def forward(self, x, mode='noise'):
        # returns dict(x_hat, likelihoods{y,z}, bits[B]);  mode in {'noise','ste','round'}
        if mode not in ('noise', 'ste', 'round'):
            raise ValueError(mode)
        y = self.g_a((x - self.norm_mean) / self.norm_std)
        z = self.h_a(y)
        if mode == 'round':
            z_hat, z_lik = self.entropy_bottleneck(z, training=False)
        else:
            z_noisy, z_lik = self.entropy_bottleneck(z, training=True)
            if mode == 'noise':
                z_hat = z_noisy
            else:
                med = self.entropy_bottleneck._get_medians()
                z_hat = ste_round(z - med) + med
        means, scales = self._hyper_params(z_hat, y.shape[-2:])
        if mode == 'round':
            y_hat, y_lik = self.gaussian_conditional(y, scales, means=means, training=False)
        else:
            y_noisy, y_lik = self.gaussian_conditional(y, scales, means=means, training=True)
            y_hat = y_noisy if mode == 'noise' else ste_round(y - means) + means
        x_hat = self.g_s(y_hat) * self.norm_std + self.norm_mean
        bits = (-torch.log2(y_lik)).flatten(1).sum(1) + (-torch.log2(z_lik)).flatten(1).sum(1)
        return {'x_hat': x_hat, 'likelihoods': {'y': y_lik, 'z': z_lik}, 'bits': bits}

    # ------------------------------------------------------- real entropy coding
    @torch.no_grad()
    def compress(self, x):
        # x: [B, C, H, W] -> list (len B) of (y_string, z_string), plus (H, W)
        y = self.g_a((x - self.norm_mean) / self.norm_std)
        z = self.h_a(y)
        z_strings = self.entropy_bottleneck.compress(z)
        z_hat = self.entropy_bottleneck.decompress(z_strings, z.shape[-2:])
        means, scales = self._hyper_params(z_hat, y.shape[-2:])
        indexes = self.gaussian_conditional.build_indexes(scales)
        y_strings = self.gaussian_conditional.compress(y, indexes, means=means)
        return list(zip(y_strings, z_strings)), tuple(y.shape[-2:])

    @torch.no_grad()
    def decompress(self, strings, hw):
        y_strings = [s[0] for s in strings]
        z_strings = [s[1] for s in strings]
        z_hat = self.entropy_bottleneck.decompress(z_strings, _z_hw(*hw))
        means, scales = self._hyper_params(z_hat, hw)
        indexes = self.gaussian_conditional.build_indexes(scales)
        y_hat = self.gaussian_conditional.decompress(y_strings, indexes, means=means)
        return self.g_s(y_hat) * self.norm_std + self.norm_mean


def load_codec(path, device=device):
    # same as feature_codec.load_codec, but builds the class defined above
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    codec = FeatureHyperpriorCodec(**ckpt['codec_config'])
    state = ckpt['codec_state']
    buffers = dict(codec.named_buffers())
    for name, value in state.items():          # CDF tables have data-dependent sizes
        if name in buffers and buffers[name].shape != value.shape:
            buffers[name].resize_(value.shape)
    codec.load_state_dict(state, strict=True)
    return codec.to(device).eval(), ckpt

The bitstream container (`feature_codec.pack_study`) adds a 4-byte header (version, number of views, H, W) and a
varint length before each string. **Every byte of it is counted in the reported rate.**

In [ ]:
print(inspect.getsource(fcodec.pack_study))

### 3.1 Shapes and parameters of a trained codec
`F_r500` = task-aware codec trained for 500 bits per image. The trace below runs each block separately on the
example feature.

In [ ]:
codec, ck = load_codec(RESULTS / 'F_r500' / 'best.pt')
print('config:', ck['codec_config'], '| trained', ck['epoch'], 'epochs | final mu =', f"{ck['mu']:.3e}")
for name in ('g_a', 'h_a', 'h_s', 'g_s', 'entropy_bottleneck'):
    print(f'  {name:20s} {n_params(getattr(codec, name)):>10,} parameters')
print(f'  {"total":20s} {n_params(codec):>10,} parameters')

with torch.no_grad():
    xn = (f_cached - codec.norm_mean) / codec.norm_std
    y = codec.g_a(xn)
    z = codec.h_a(y)
    z_hat, z_lik = codec.entropy_bottleneck(z, training=False)
    means, scales = codec._hyper_params(z_hat, y.shape[-2:])
    y_hat, y_lik = codec.gaussian_conditional(y, scales, means=means, training=False)
    x_hat = codec.g_s(y_hat) * codec.norm_std + codec.norm_mean
for label, t in [('input feature x', f_cached), ('y = g_a(x)', y), ('z = h_a(y)', z),
                 ('mean, scale = h_s(z_hat)', means), ('x_hat = g_s(y_hat)', x_hat)]:
    print(f'{label:26s} {str(tuple(t.shape)):20s}')
y_bits = (-torch.log2(y_lik)).sum().item(); z_bits = (-torch.log2(z_lik)).sum().item()
print(f'\nestimated bits: y {y_bits:.0f} + z {z_bits:.0f} = {y_bits + z_bits:.0f}')
print(f'nonzero rounded y symbols: {int((torch.round(y - means) != 0).sum())} of {y.numel()}')

### 3.2 Real entropy coding
`compress` produces actual bytes (range coder), `pack_study` wraps them, and the receiver unpacks and decodes.
The decoded feature must equal `forward(mode='round')`, and the actual size includes the container and
coder flush overhead (about 20 bytes per image at this rate, which is why the training rate controller targets the
*actual* size, Section 5.4).

In [ ]:
strings, hw = codec.compress(f_cached)     # CDF tables were loaded with the checkpoint
blob = fcodec.pack_study(strings, hw)                 # what would be transmitted / stored
views_rx, hw_rx = fcodec.unpack_study(blob)
f_dec = codec.decompress(views_rx, hw_rx)
with torch.no_grad():
    f_round = codec(f_cached, mode='round')['x_hat']

print(f'y string {len(strings[0][0])} B, z string {len(strings[0][1])} B, file {len(blob)} B = {8*len(blob)} bits')
print(f'compression ratio: {RAW_BITS / (8*len(blob)):,.0f} x')
print(f'max |decoded - forward(round)| = {(f_dec - f_round).abs().max().item():.2e}')
print(f'feature NMSE = {tc.feature_nmse(f_dec, f_cached).item():.3f}')

# the project module and the copy above give the same result
ref_codec, _ = fcodec.load_codec(RESULTS / 'F_r500' / 'best.pt', device)
with torch.no_grad():
    diff = (ref_codec(f_cached, mode='round')['x_hat'] - f_round).abs().max().item()
print(f'max difference to feature_codec.py: {diff:.1e}')

## 4. Reports from compressed features
Every codec checkpoint is used through the deployed path (bytes -> decode -> frozen R2Gen beam search). The reports
should match the ones saved by `evaluate.py` (which decoded in batches of 16; GPU rounding can rarely change a word).

In [ ]:
def transmit(codec, f):
    strings, hw = codec.compress(f)
    blob = fcodec.pack_study(strings, hw)
    views_rx, hw_rx = fcodec.unpack_study(blob)
    return codec.decompress(views_rx, hw_rx), 8 * len(blob)

labs_A = abnormal(cs.report_labels(report_A.split()))
print(f'A   uncompressed ({RAW_BITS:,} bits)\n    {report_A}\n    abnormal: {sorted(labs_A)}\n')
live = {}
for rate in (500, 1000, 2000):
    saved = {c: {r['id']: r for r in json.loads((RESULTS / f'eval_test_r{rate}' / 'per_study.json')
                                                .read_text(encoding='utf-8'))[c]}[EX_ID] for c in ('Bft', 'F')}
    for name in ('Bft', 'F'):
        c, _ = load_codec(RESULTS / f'{name}_r{rate}' / 'best.pt')
        f_hat, bits = transmit(c, f_cached)
        rep = r2.generate(f_hat, None)[0]
        live[(name, rate)] = rep
        labs = abnormal(cs.report_labels(rep.split()))
        flag = 'same findings as A' if labs == labs_A else 'DIFFERENT findings'
        print(f'{name:3s} {bits:5d} bits  [{flag}]  identical to saved: {rep == saved[name]["generated"]}')
        print(f'    {rep}\n    abnormal: {sorted(labs)}\n')

### 4.1 More examples from the saved 500-bit test evaluation
No computation; read from `results/mimic_full/eval_test_r500/per_study.json`. For each study: the radiologist's
abnormal findings (rule labeller), and what A, Bft and F report.

In [ ]:
import random
random.seed(3)
show = random.sample(candidates, 3) + random.sample(
    [i for i in test_store.ids if abnormal(saved500['Bft'][i]['labels']) == abnormal(saved500['A'][i]['labels'])
     and abnormal(saved500['F'][i]['labels']) != abnormal(saved500['A'][i]['labels'])], 2)
for i in show:
    ref = saved500['A'][i]['reference']
    print('=' * 110)
    print('RADIOLOGIST:', ref[:300], '...' if len(ref) > 300 else '')
    print('   abnormal:', sorted(abnormal(cs.report_labels(ref.split()))))
    for c in ('A', 'Bft', 'F'):
        r = saved500[c][i]
        bits = 'uncompressed' if c == 'A' else f"{r['bits']} bits"
        print(f'{c:3s} ({bits}): {r["generated"][:300]}')
        print('   abnormal:', sorted(abnormal(r['labels'])))
print('=' * 110)
print('The last two are cases where F changes the findings and Bft does not; both directions happen.')

## 5. Training objective of the task-aware codec F

$$
L \;=\; \mu \cdot R \;+\; \lambda_f\,\mathrm{NMSE}(\hat x, x) \;+\; \lambda_t\,\mathrm{NLL}_w(\hat y^{A}\mid \hat x)
\;+\; \lambda_{kd}\,\mathrm{KL}\big(p(\cdot\mid x)\,\|\,p(\cdot\mid \hat x)\big)
$$

* $R$: estimated bits per image, $-\log_2$ of the entropy-model likelihoods of $y$ and $z$.
* NMSE: feature error normalised by the feature energy (the only distortion term of Bft).
* $\mathrm{NLL}_w$: teacher-forced negative log-likelihood, through the frozen decoder, of $\hat y^{A}$ = **the report
  R2Gen generated from the uncompressed feature** (`make_teacher_reports.py`). Tokens inside clinical spans get weight
  $1+\alpha = 3$, the rest weight 1.
* KL: at every position of that report, the next-token distribution from the compressed feature is pulled toward
  the distribution from the uncompressed feature (whole vocabulary, not just the chosen token).
* MIMIC settings: $\lambda_f=1$, $\lambda_t=1$, $\lambda_{kd}=100$, $\alpha=2$; $\mu$ is set by the rate controller.

Every text term is zero for a lossless codec, so F can only approach A, never beat it.

### 5.1 The loss terms (copied from `ta_common.py` and `train_codec_distill.py`)

In [ ]:
def feature_nmse(f_hat, f, eps=1e-8):
    # mean((F_hat - F)^2 / (mean(F^2) + eps)), energy per sample
    err = (f_hat - f).square()
    energy = f.square().flatten(1).mean(1).view(-1, 1, 1, 1)
    return (err / (energy + eps)).mean()


def weighted_nll(logprobs, targets, masks, weights=None):
    # sum(mask * w * nll) / sum(mask * w), with R2Gen's shift (predict token t+1 from tokens <= t)
    T = logprobs.shape[1]
    tgt = targets[:, 1:][:, :T]
    m = masks[:, 1:][:, :T].to(logprobs.dtype)
    if weights is not None:
        m = m * weights[:, :T].to(logprobs.dtype)
    nll = -logprobs.gather(2, tgt.long().unsqueeze(2)).squeeze(2)
    return (nll * m).sum() / m.sum().clamp_min(1e-8)


def distillation_kl(student_logprobs, teacher_logprobs, masks):
    # KL(teacher || student) averaged over valid next-token positions; teacher detached
    T = student_logprobs.shape[1]
    mask = masks[:, 1:][:, :T].to(student_logprobs.dtype)
    log_p = torch.log_softmax(teacher_logprobs.detach(), dim=-1)
    log_q = torch.log_softmax(student_logprobs, dim=-1)
    terms = (log_p.exp() * (log_p - log_q)).sum(dim=-1)
    return (terms * mask).sum() / mask.sum()

### 5.2 Which tokens are up-weighted
`clinical_spans.py` finds mentions of 18 findings (cardiomegaly, effusion, pneumothorax, edema, ...) with their
negation / uncertainty cue and adjacent modifiers ("no **left pleural effusion**", "cannot exclude **edema**") and
marks the whole span. Below, a training study's teacher report with its weight-3 tokens in **[brackets]**.

In [ ]:
print('concepts:', ', '.join(cs.CONCEPTS))
teacher = json.loads((RESULTS / 'teacher_reports.json').read_text(encoding='utf-8'))
train_store = tc.FeatureStore(FEAT, 'train')
ALPHA = 2.0

def show_weights(text):
    words = tokenizer.clean_report(text).split()
    n_t = len(tokenizer(text)[:args.max_seq_length]) - 1
    w, info = cs.token_weights(words, n_t, ALPHA)
    out = [f'[{wd}]' if j < len(w) and w[j] > 1 else wd for j, wd in enumerate(words)]
    print(' '.join(out))
    print(f"-> {info['upweighted_tokens']} of {info['n_targets']} target tokens weighted {1 + ALPHA:g}, "
          f"labels: {cs.report_labels(words)}\n")

shown = 0
for sid in train_store.ids:
    rep = teacher['reports']['train'][sid]
    if abnormal(cs.report_labels(rep.split())):
        show_weights(rep); shown += 1
    if shown == 2:
        break

### 5.3 A few real training steps
This is the inner loop of `train_codec_distill.py` (called by `train_codec_fidelity.py`, which swaps the targets
for the teacher reports), run for a few steps on a small slice of the training set. It starts from the 500-bit
feature-only codec `B_r500`, exactly as F was initialised. A copy is trained here; nothing is saved.

In [ ]:
from train_codec_fidelity import substitute_reports

N_TRAIN = 256
STEPS = 40 if device.type == 'cuda' else 4
BATCH = 16
CFG = dict(target_bits=500, lambda_f=1.0, lambda_t=1.0, lambda_kd=100.0, lr=5e-5, aux_lr=1e-3, mu_lr=0.02, clip=1.0)

train_ids = train_store.ids[:N_TRAIN]
ann_teacher = substitute_reports(ann, teacher['reports'], splits=('train',), required={'train': train_ids})
bank = tc.TargetBank(ann_teacher, 'train', tokenizer, args.max_seq_length, ALPHA, train_ids)

student, ck_B = load_codec(RESULTS / 'B_r500' / 'best.pt')
student.train()
main_p, aux_p = student.trainable_groups()
opt = torch.optim.Adam(main_p, lr=CFG['lr'])
aux_opt = torch.optim.Adam(aux_p, lr=CFG['aux_lr'])
log_mu = math.log(ck_B['mu'])                                     # trainer reuses B's mu at the same target
overhead = ck_B['val']['actual_bits_per_study'] - ck_B['val']['est_bits_per_study']   # container + coder flush
print(f"start mu = {math.exp(log_mu):.3e}, container overhead ~ {overhead:.0f} bits")


def train_step(idx):
    global log_mu
    x = train_store.batch(idx, device)                            # [B, 1, 2048, 7, 7]
    xf = x.flatten(0, 1)
    out = student(xf, mode='noise')                               # uniform-noise quantisation
    bits = out['bits'].view(x.shape[0], x.shape[1]).sum(1).mean()
    d_f = feature_nmse(out['x_hat'], xf)
    t, m, w = bank.batch(idx, device)                             # teacher-report tokens, mask, span weights
    lp = r2.teacher_forced(*tc.views(out['x_hat'].view_as(x)), t) # student log-probs through frozen decoder
    wnll = weighted_nll(lp, t, m, w)
    with torch.no_grad():
        lp_teacher = r2.teacher_forced(*tc.views(x), t)           # uncompressed log-probs
    kd = distillation_kl(lp, lp_teacher, m)
    loss = math.exp(log_mu) * bits + CFG['lambda_f'] * d_f + CFG['lambda_t'] * wnll + CFG['lambda_kd'] * kd

    opt.zero_grad(set_to_none=True)
    loss.backward()
    assert all(p.grad is None for p in r2.model.parameters()), 'R2Gen must stay frozen'
    torch.nn.utils.clip_grad_norm_(main_p, CFG['clip'])
    opt.step()
    aux_opt.zero_grad(set_to_none=True)
    student.aux_loss().backward()                                 # fits the z-density quantiles
    aux_opt.step()

    # rate controller: push mu up if the ROUNDED (deployed) rate + overhead is above target, down if below
    with torch.no_grad():
        bits_round = student(xf, mode='round')['bits'].view(x.shape[0], x.shape[1]).sum(1).mean().item()
    log_mu += CFG['mu_lr'] * math.log(max(bits_round + overhead, 1.0) / CFG['target_bits'])
    return dict(loss=loss.item(), bits=bits.item(), bits_round=bits_round, nmse=d_f.item(),
                wnll=wnll.item(), kl=kd.item(), mu=math.exp(log_mu))


rng = np.random.default_rng(0)
log = []
t0 = time.time()
for step in range(STEPS):
    log.append(train_step(rng.choice(N_TRAIN, BATCH, replace=False).tolist()))
    if step % max(1, STEPS // 8) == 0 or step == STEPS - 1:
        print(f"step {step:3d} " + ' '.join(f'{k}={v:.4g}' for k, v in log[-1].items()))
print(f'{STEPS} steps in {time.time() - t0:.0f}s')

### 5.4 How the full runs were trained (`run_mimic.sh`)
| codec | start | epochs | lr | loss |
|---|---|---|---|---|
| **B** | scratch (normalisation from train split) | 30 | 1e-4 | rate + NMSE |
| **Bft** | B | +10 | 5e-5 | rate + NMSE (same budget as F, fair comparison) |
| **F** | B | +10 | 5e-5 | rate + NMSE + weighted NLL + 100 x KL, targets = teacher reports |

12,000 training images, batch 16, Adam with cosine decay, gradient clipping 1.0. The rate controller updates
$\log\mu \mathrel{+}= 0.02\,\log\frac{\text{rounded bits}+\text{overhead}}{\text{target}}$ every step, and the
checkpoint kept is the best validation epoch whose **actual** file size is within 10% of the target. One run per
codec and rate (500, 1,000, 2,000 bits). Validation curves of the real runs:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for name, color in (('Bft', '#2a78d6'), ('F', '#1baf7a')):
    for rate, ls in ((500, '-'), (1000, '--'), (2000, ':')):
        h = json.loads((RESULTS / f'{name}_r{rate}' / 'history.json').read_text())
        ep = [r['epoch'] for r in h if 'val' in r]
        v = [r['val'] for r in h if 'val' in r]
        axes[0].plot(ep, [x['actual_bits_per_study'] for x in v], ls, color=color, label=f'{name} {rate}')
        axes[1].plot(ep, [x['text_nll'] for x in v], ls, color=color)
        axes[2].plot(ep, [x['feature_nmse'] for x in v], ls, color=color)
axes[0].set_title('actual bits per image (val)'); axes[1].set_title('NLL of target report (val)')
axes[2].set_title('feature NMSE (val)')
for a in axes:
    a.set_xlabel('epoch (0 = start from B)'); a.grid(alpha=.3)
axes[0].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()
print('Note: Bft is validated on the radiologist report and F on the teacher report, so NLL levels differ between them.')

## 6. Results on the MIMIC-CXR test split (3,858 images)
Read from the saved evaluations (`evaluate.py`, `check_labeller_vs_chexpert.py`); intervals are 95% paired
bootstrap over studies.

* **Abnormal-finding F1 vs CheXpert labels**: findings in the generated report (rule labeller) compared with the
  official CheXpert labels of the radiologist's report. Measures clinical accuracy.
* **Same findings as A**: share of images whose set of abnormal findings is identical to the uncompressed report.
  Measures how well the codec preserves what the model would have said.

In [ ]:
chex = json.loads((RESULTS / 'chexpert_check.json').read_text())['codec']
fid = json.loads((RESULTS / 'fidelity_stats.json').read_text())

print(f"{'bits':>5} | {'F1 A':>6} {'F1 Bft':>7} {'F1 F':>6} {'F-Bft [95% CI]':>24} | "
      f"{'same findings Bft':>17} {'F':>6} | {'identical Bft':>13} {'F':>6}")
for split in ('test', 'val'):
    print(f'--- {split}')
    for t in (500, 1000, 2000):
        c, s = chex[f'{split}_{t}'], fid[f'{split}_{t}']
        d = c['F_minus_Bft']
        print(f"{t:5d} | {c['A']:6.3f} {c['Bft']:7.3f} {c['F']:6.3f} {d[0]:+8.3f} [{d[1]:+.3f}, {d[2]:+.3f}] | "
              f"{s['Bft']['findings']:16.1f}% {s['F']['findings']:5.1f}% | "
              f"{s['Bft']['identical']:12.1f}% {s['F']['identical']:5.1f}%")
print(f'\nuncompressed feature = {RAW_BITS:,} bits per image; actual codec rates:')
for t in (500, 1000, 2000):
    print(f"  target {t}: Bft {fid[f'test_{t}']['Bft']['bits']:.0f}, F {fid[f'test_{t}']['F']['bits']:.0f} bits")

In [ ]:
T = (500, 1000, 2000)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, color, label in (('Bft', '#2a78d6', 'feature-only codec (Bft)'),
                           ('F', '#1baf7a', 'task-aware codec (F)')):
    xs = [fid[f'test_{t}'][name]['bits'] for t in T]
    axes[0].plot(xs, [chex[f'test_{t}'][name] for t in T], 'o-', color=color, label=label)
    axes[1].plot(xs, [fid[f'test_{t}'][name]['findings'] for t in T], 'o-', color=color, label=label)
axes[0].axhline(chex['test_500']['A'], ls='--', color='gray', label='uncompressed R2Gen')
axes[1].axhline(100, ls='--', color='gray', label='uncompressed R2Gen')
axes[0].set_ylabel('abnormal-finding F1 vs CheXpert labels')
axes[1].set_ylabel('same findings as uncompressed (%)')
for a in axes:
    a.set_xscale('log', base=2); a.set_xticks(T); a.set_xticklabels(['500', '1,000', '2,000'])
    a.set_xlabel('actual bits per image'); a.grid(alpha=.3); a.legend(fontsize=8)
fig.suptitle('MIMIC-CXR test, frozen R2Gen')
plt.tight_layout(); plt.show()

**Reading the results.**
* At **500 bits** F recovers most of the clinical accuracy lost by the feature-only codec
  (F1 0.268 vs 0.218; uncompressed 0.281) and keeps the uncompressed findings for 44% of images vs 29%.
* The gain shrinks as the rate grows (+0.015 at 1,000 bits, +0.008 at 2,000 bits), because the feature-only codec
  already preserves most findings there. At 2,000 bits F is indistinguishable from uncompressed in F1
  (+0.003, CI -0.005 to +0.011).
* On the validation split the F1 gain is clear only at 500 bits; the same-findings gain is clear at all rates.
* **Do not** read this as "same accuracy at half the bits": by CheXpert F1, F at 535 bits is close to Bft at 1,020
  bits, but by same-findings it is not (44.1% vs 49.6%).
* Limitations: one training run per codec on MIMIC; findings come from a rule labeller (micro-F1 0.67 against
  CheXpert on radiologist reports); R2Gen itself has low recall (it mentions about one in five radiologist findings).

## 7. Where things can be changed
| what | where | notes |
|---|---|---|
| codec architecture (e.g. more channels, ELIC-style context model, other feature codecs) | `feature_codec.py` (`FeatureHyperpriorCodec`) | trainers only need `forward(mode)`, `compress`, `decompress`, `trainable_groups`, `aux_loss`, `update` |
| latent size / capacity | `--c_y` in `run_mimic.sh` (64 for MIMIC) | |
| target rates | `TARGETS="250 500 1000" bash task_aware/run_mimic.sh` | actual-size rate control |
| loss weights | `--lambda_t`, `--lambda_kd`, `--alpha`, `--lambda_f` | ablations: KL only, NLL only, no span weights |
| which tokens matter | `clinical_spans.py` (`CONCEPTS`, cues, `NORMAL_RULES`) | could be replaced by RadGraph / CheXbert spans |
| training target | `make_teacher_reports.py` -> `train_codec_fidelity.py` | e.g. teacher with sampling, n-best lists |
| evaluation labeller | `evaluate.py`, `check_labeller_vs_chexpert.py` | CheXbert would give a stronger clinical metric |
| report generator | `ta_common.FrozenR2Gen` | any model with a teacher-forced and a generate call can be plugged in |
| baselines | `evaluate.py --cond NAME=path` | e.g. image codecs (JPEG/VVC) before the CNN |

### Files
| file | role |
|---|---|
| `task_aware/extract_features.py` | cache ResNet layer4 features for train/val/test |
| `task_aware/make_teacher_reports.py` | uncompressed R2Gen reports = training targets of F |
| `task_aware/train_codec.py` | trains B and Bft (rate + NMSE, optional reference-report NLL) |
| `task_aware/train_codec_distill.py` | adds the KL distillation term |
| `task_aware/train_codec_fidelity.py` | replaces the targets by the teacher reports and runs the distillation trainer |
| `task_aware/evaluate.py` | deployed-path evaluation: bytes on disk -> decode -> beam search -> metrics, bootstrap |
| `task_aware/run_mimic.sh` | the whole MIMIC experiment (resumable) |
| `check_labeller_vs_chexpert.py`, `analyze_mimic_reports.py`, `plot_mimic.py` | CheXpert check, per-finding analysis, figures |